In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

tharrineeshs_headpho_path = kagglehub.dataset_download('tharrineeshs/headpho')
tharrineeshs_semeval_laptops_path = kagglehub.dataset_download('tharrineeshs/semeval-laptops')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/semeval-laptops/Laptop_Train_v2.csv")
df = df.dropna(subset=["Sentence", "Aspect Term", "polarity"]).reset_index(drop=True)

# 3‑class mapping (no conflict)
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

df = df[df["polarity"].isin(label2id.keys())].reset_index(drop=True)
df["label"] = df["polarity"].map(label2id)

def build_text(row):
    return f"aspect: {row['Aspect Term']} [SEP] sentence: {row['Sentence']}"

df["text"] = df.apply(build_text, axis=1)


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


In [ ]:
neutral_id = label2id["neutral"]
neutral_df = train_df[train_df["label"] == neutral_id]

# Simple oversampling: duplicate neutral once
train_df_bal = pd.concat([train_df, neutral_df]).sample(frac=1, random_state=42).reset_index(drop=True)

train_texts = train_df_bal["text"].tolist()
train_labels = train_df_bal["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()


In [ ]:
from transformers import AutoTokenizer
import torch

model_name = "roberta-base"  # or your current base
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts, labels, max_len=128):
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

train_enc = tokenize(train_texts, train_labels, max_len=128)
test_enc = tokenize(test_texts, test_labels, max_len=128)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class ABSADataset(Dataset):
    def __init__(self, enc):
        self.enc = enc
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.enc.items()}

train_ds = ABSADataset(train_enc)
test_ds = ABSADataset(test_enc)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)


In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_labels = len(label2id)
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)

# Class weights on the (oversampled) train set:
classes = np.array(sorted(df["label"].unique()))  # use original df distribution
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(base_model.parameters(), lr=2e-5)
num_epochs = 4  # a bit more than 3
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def train_epoch():
    base_model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits
        loss = loss_fct(logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

for epoch in range(num_epochs):
    train_loss = train_epoch()
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}")


Epoch 1: train_loss=0.8405
Epoch 2: train_loss=0.4673
Epoch 3: train_loss=0.2858
Epoch 4: train_loss=0.1760


In [ ]:
from sklearn.metrics import classification_report

def evaluate(loader):
    base_model.eval()
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy().tolist()

            outputs = base_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            preds = outputs.logits.argmax(dim=-1).cpu().numpy().tolist()

            all_labels.extend(labels)
            all_preds.extend(preds)

    print(classification_report(
        all_labels,
        all_preds,
        target_names=[id2label[i] for i in range(num_labels)]
    ))

evaluate(test_loader)


              precision    recall  f1-score   support

    negative       0.88      0.80      0.84       173
     neutral       0.64      0.77      0.70        92
    positive       0.87      0.85      0.86       198

    accuracy                           0.82       463
   macro avg       0.80      0.81      0.80       463
weighted avg       0.83      0.82      0.82       463



In [ ]:
def predict_aspect_sentiment(sentence, aspect):
    text = f"aspect: {aspect} [SEP] sentence: {sentence}"
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    base_model.eval()
    with torch.no_grad():
        out = base_model(**enc)
        pred_id = out.logits.argmax(dim=-1).item()
    return id2label[pred_id]


In [ ]:
import pandas as pd
from collections import defaultdict

lap_df = pd.read_csv("/kaggle/input/semeval-laptops/Laptop_Train_v2.csv")
lap_df = lap_df.dropna(subset=["Sentence", "Aspect Term"]).reset_index(drop=True)

def merge_overlapping_spans(spans):
    spans = sorted(spans, key=lambda x: (x["start"], -x["end"] - x["start"]))
    merged = []
    for s in spans:
        if not merged:
            merged.append(s)
            continue
        last = merged[-1]
        if s["start"] < last["end"] and s["end"] > last["start"]:
            len_last = last["end"] - last["start"]
            len_s = s["end"] - s["start"]
            if len_s > len_last:
                merged[-1] = s
        else:
            merged.append(s)
    return merged

tmp_groups = defaultdict(list)
for _, row in lap_df.iterrows():
    s = str(row["Sentence"])
    a = str(row["Aspect Term"])
    start = int(row["from"])
    end = int(row["to"])
    tmp_groups[s].append({"term": a, "start": start, "end": end})

sent_list = []
for s, spans in tmp_groups.items():
    merged_spans = merge_overlapping_spans(spans)
    sent_list.append({"sentence": s, "spans": merged_spans})

print("Num sentences for AE:", len(sent_list))


Num sentences for AE: 1482


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

bert_ae_name = "bert-base-uncased"
tag_tokenizer = AutoTokenizer.from_pretrained(bert_ae_name)

tag2id = {"O": 0, "B-ASP": 1, "I-ASP": 2}
id2tag = {v: k for k, v in tag2id.items()}

class AspectTagDataset(Dataset):
    def __init__(self, sent_list):
        self.sent_list = sent_list
    def __len__(self):
        return len(self.sent_list)
    def __getitem__(self, idx):
        item = self.sent_list[idx]
        return {
            "sentence": item["sentence"],
            "spans": item["spans"],
        }

def encode_with_bio(batch, max_len=128):
    sentences = [b["sentence"] for b in batch]
    spans_batch = [b["spans"] for b in batch]

    encodings = tag_tokenizer(
        sentences,
        return_offsets_mapping=True,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    all_labels = []
    for offsets, spans in zip(encodings["offset_mapping"], spans_batch):
        labels = ["O"] * len(offsets)
        for span in spans:
            s = span["start"]
            e = span["end"]
            first = True
            for i, (start_char, end_char) in enumerate(offsets.tolist()):
                if start_char >= e or end_char <= s:
                    continue
                if first:
                    labels[i] = "B-ASP"
                    first = False
                else:
                    labels[i] = "I-ASP"
        all_labels.append([tag2id[l] for l in labels])

    labels_tensor = torch.tensor(all_labels, dtype=torch.long)
    encodings.pop("offset_mapping")
    encodings["labels"] = labels_tensor
    return encodings

def collate_fn(batch):
    return encode_with_bio(batch, max_len=128)

full_ae_dataset = AspectTagDataset(sent_list)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

indices = list(range(len(full_ae_dataset)))
ae_train_idx, ae_val_idx = train_test_split(indices, test_size=0.2, random_state=42)

ae_train_dataset = Subset(full_ae_dataset, ae_train_idx)
ae_val_dataset = Subset(full_ae_dataset, ae_val_idx)

ae_train_loader = DataLoader(ae_train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
ae_val_loader = DataLoader(ae_val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [ ]:
from transformers import AutoModelForTokenClassification, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tag_model = AutoModelForTokenClassification.from_pretrained(
    bert_ae_name,
    num_labels=len(tag2id)
).to(device)

optimizer_ae = torch.optim.AdamW(tag_model.parameters(), lr=3e-5)
ae_num_epochs = 3
ae_num_training_steps = ae_num_epochs * len(ae_train_loader)
ae_scheduler = get_linear_schedule_with_warmup(
    optimizer_ae,
    num_warmup_steps=int(0.1 * ae_num_training_steps),
    num_training_steps=ae_num_training_steps
)

def train_ae_epoch():
    tag_model.train()
    total_loss = 0.0
    for batch in ae_train_loader:
        optimizer_ae.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = tag_model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer_ae.step()
        ae_scheduler.step()
        total_loss += loss.item()
    return total_loss / len(ae_train_loader)

def eval_ae_epoch():
    tag_model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in ae_val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = tag_model(**batch)
            total_loss += outputs.loss.item()
    return total_loss / len(ae_val_loader)

for epoch in range(ae_num_epochs):
    tr = train_ae_epoch()
    vl = eval_ae_epoch()
    print(f"[AE] Epoch {epoch+1}: train_loss={tr:.4f}, val_loss={vl:.4f}")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[AE] Epoch 1: train_loss=0.2602, val_loss=0.0750
[AE] Epoch 2: train_loss=0.0639, val_loss=0.0493
[AE] Epoch 3: train_loss=0.0392, val_loss=0.0479


In [ ]:
def extract_aspects_bert_raw_spans(sentence, max_len=128):
    enc = tag_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    tag_model.eval()
    with torch.no_grad():
        logits = tag_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

    pred_ids = logits.argmax(dim=-1)[0].cpu().tolist()
    tokens = tag_tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

    aspects = []
    current = []
    for tok, tid in zip(tokens, pred_ids):
        tag = id2tag[tid]
        if tag == "B-ASP":
            if current:
                aspects.append(current)
                current = []
            current.append(tok)
        elif tag == "I-ASP" and current:
            current.append(tok)
        else:
            if current:
                aspects.append(current)
                current = []
    if current:
        aspects.append(current)

    phrases = []
    for toks in aspects:
        phrase = tag_tokenizer.convert_tokens_to_string(toks)
        phrase = phrase.replace(" ##", "").strip()
        phrases.append(phrase)
    return phrases

# quick sanity check
ex = "I charge it at night and skip taking the cord with me because of the good battery life."
print(extract_aspects_bert_raw_spans(ex))


['charge', 'cord', 'battery life']


In [ ]:
!pip install rapidfuzz
import spacy
spacy_nlp = spacy.load("en_core_web_sm")

GENERIC_PREFIXES = {"this", "that", "my", "your", "the"}

from rapidfuzz import process, fuzz

# train_aspects was built from Laptop_Train_v2 earlier
# train_aspects = list of lowercased aspect terms from training

def map_to_known_aspect(extracted, known_terms, threshold=85):
    cand, score, _ = process.extractOne(
        extracted.lower(),
        known_terms,
        scorer=fuzz.token_set_ratio
    )
    if score >= threshold:
        return cand
    return extracted.lower()

manual_map = {
    "glass touchpad": "touchpad",
    "hard drive light": "hard drive",
}

def apply_manual_map(a):
    return manual_map.get(a, a)

def extract_canonical_aspects(sentence, max_len=128, threshold=85):
    raw_spans = extract_aspects_bert_raw_spans(sentence, max_len=max_len)

    cleaned = []
    for p in raw_spans:
        pl = p.lower().strip()
        if len(pl.split()) <= 2:
            cleaned.append(pl)
        else:
            doc = spacy_nlp(pl)
            content = [t.text for t in doc if t.text not in GENERIC_PREFIXES]
            cleaned.append(" ".join(content) if content else pl)

    canonical = []
    seen = set()
    for c in cleaned:
        mapped = map_to_known_aspect(c, train_aspects, threshold=threshold)
        mapped = apply_manual_map(mapped)
        if mapped not in seen:
            seen.add(mapped)
            canonical.append(mapped)
    return canonical



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
s = "I charge it at night and skip taking the cord with me because of the good battery life."
print(extract_aspects_bert_raw_spans(s))      # ['charge','cord','battery life']
print(extract_canonical_aspects(s))           # e.g. ['charge','cord','battery life'] but now aligned to training vocab


['charge', 'cord', 'battery life']
['charge time', 'cord', 'battery life']


In [ ]:
core_aspects = {
    "battery life", "battery",
    "screen", "display",
    "keyboard", "keys",
    "touchpad", "glass touchpad", "trackpad",
    "speaker", "speakers",
    "sound", "audio",
    "performance", "speed",
    "price", "value",
    "quality", "build quality",
    "fan", "noise",
    "graphics","graphics card", "gpu",
    "wifi", "wireless",
    "operating system", "system", "os",
    "hard drive", "drive", "storage",
    "warranty", "service", "warranty service"
}
  # extend as needed

def filter_core(aspects):
    return [a for a in aspects if a in core_aspects]


In [ ]:
def analyze_sentence(sentence, threshold=85, use_core_filter=False):
    aspects = extract_canonical_aspects(sentence, threshold=threshold)
    if use_core_filter:
        aspects = filter_core(aspects)
    results = []
    for asp in aspects:
        sent = predict_aspect_sentiment(sentence, asp)
        results.append({"aspect": asp, "sentiment": sent})
    return results

def print_sentence_analysis(sentence, threshold=85, use_core_filter=False):
    res = analyze_sentence(sentence, threshold=threshold, use_core_filter=use_core_filter)
    print("Sentence:", sentence)
    for r in res:
        print(f"  - Aspect: {r['aspect']} | Sentiment: {r['sentiment']}")


In [ ]:
s = "I charge it at night and skip taking the cord with me because of the good battery life."
print_sentence_analysis(s, use_core_filter=True)


Sentence: I charge it at night and skip taking the cord with me because of the good battery life.
  - Aspect: battery life | Sentiment: positive


In [ ]:
s1 = "The battery life is amazing but the screen is too dim and the keyboard feels cheap."
s2 = "The laptop is fast and the performance is great, but the fan is very noisy."
s3 = "The screen is bright and sharp, but the battery dies in just two hours."
s4 = "I love the keyboard and trackpad, but the speakers are disappointing."
s5 = "The build quality is solid and the price is reasonable."
s6 = "The wifi keeps disconnecting and the operating system crashes frequently."
s7 = "For this price, the performance and graphics are excellent."
s8 = "The touchpad is unresponsive and the keys sometimes get stuck."
s9 = "The laptop runs quietly and stays cool even under heavy load."
s10 = "The warranty service was slow, but they eventually replaced my hard drive."

for s in [s1, s2, s3, s4, s5, s6, s7, s8, s9, s10]:
    print_sentence_analysis(s, use_core_filter=True)
    print()


Sentence: The battery life is amazing but the screen is too dim and the keyboard feels cheap.
  - Aspect: battery life | Sentiment: positive
  - Aspect: screen | Sentiment: negative
  - Aspect: keyboard | Sentiment: negative

Sentence: The laptop is fast and the performance is great, but the fan is very noisy.
  - Aspect: performance | Sentiment: positive
  - Aspect: fan | Sentiment: negative

Sentence: The screen is bright and sharp, but the battery dies in just two hours.
  - Aspect: screen | Sentiment: positive
  - Aspect: battery life | Sentiment: negative

Sentence: I love the keyboard and trackpad, but the speakers are disappointing.
  - Aspect: keyboard | Sentiment: positive
  - Aspect: trackpad | Sentiment: positive
  - Aspect: speakers | Sentiment: negative

Sentence: The build quality is solid and the price is reasonable.
  - Aspect: quality | Sentiment: positive
  - Aspect: price | Sentiment: positive

Sentence: The wifi keeps disconnecting and the operating system crashes f

In [ ]:
s11="The graphics look amazing, but it takes a heavy toll on the battery"
print_sentence_analysis(s11, use_core_filter=True)

Sentence: The graphics look amazing, but it takes a heavy toll on the battery
  - Aspect: graphics card | Sentiment: positive
  - Aspect: battery life | Sentiment: negative
